## Numerical Methods: Finding Roots of Equations

Roots of equations are the values of the variable that satisfy the equation, making it equal to zero. Finding roots is a fundamental problem in numerical analysis and has applications in various fields such as physics, engineering, and finance.

Key concepts in numerical methods for finding roots include:

- **Error**: Represents the deviation between the expected or actual value and the computed value. In numerical methods, we generally want to make the error as small as possible.

- **Tolerance**: The amount of error that we are willing to accept.

- **Sign**: Represents whether the value of a function is positive (+), negative (−), or zero at a particular point. In root-finding methods, the sign of the function helps us determine whether a root lies between two points.

In [521]:
import numpy as np
import scipy as sp
import sympy as sym
import math

In [522]:
# Helper function
def solve(fx_string, param1, param2, tol, method):
    # Clean for eval (NumPy context)
    fx_eval = fx_string.replace("^", "**")
    func = eval("lambda x: " + fx_eval)

    print(f"Solving '{fx_string}' using {method.__name__} method:")

    if method.__name__ == 'newton_raphson':
        x0 = param1
        # Clean for SymPy (remove 'np.' prefixes)
        fx_sympy = fx_eval.replace("np.", "")
        x_sym = sym.symbols('x')
        fx_sym = sym.sympify(fx_sympy)
        dfx_sym = sym.diff(fx_sym, x_sym)
        df_func = sym.lambdify(x_sym, dfx_sym, 'numpy')

        solution = method(func, df_func, x0, tol)
    else:
        a, b = param1, param2
        solution = method(func, a, b, tol)

    print("\nSolution:", solution)

### 1. Incremental Search Method

Incremental search is a simple method to find roots of a function by evaluating the function at regular intervals and looking for sign changes.

Formula: 

- Start with an initial guess $x$ and incrementally evaluate the function $f(x)$ at points $x + dx$, where $dx$ is a small step size. 

- If $f(x)$ and $f(x + dx)$ have opposite signs, then a root exists in the interval $[x, x + dx]$.

In [523]:
def incremental_search(f, a, b, final_dx):
    dx = 0.1  # Start with an initial dx
    while dx >= final_dx:
        x1 = a
        while x1 + dx <= b:
            x2 = x1 + dx
            f1, f2 = f(x1), f(x2)
            if np.sign(f1) != np.sign(f2):
                print(f"dx = {dx}; Interval: [{x1}, {x2}]")
                # Assign with the new interval for the next iteration
                a, b = x1, x2
                break
            x1 += dx
        dx /= 10  # Reduce the dx
    return (a + b) / 2.0

In [524]:
fx = "x^3 - x - 1"
a, b = 0, 2
dx = 0.0001

solve(fx, a, b, dx, incremental_search)

Solving 'x^3 - x - 1' using incremental_search method:
dx = 0.1; Interval: [1.3, 1.4000000000000001]
dx = 0.01; Interval: [1.32, 1.33]
dx = 0.001; Interval: [1.3239999999999996, 1.3249999999999995]
dx = 0.0001; Interval: [1.3246999999999995, 1.3247999999999995]

Solution: 1.3247499999999994


In [525]:
fx = "4 * x^3 - 2 * x - 5"
a, b = 1, 5
dx = 0.0001

solve(fx, a, b, dx, incremental_search)

Solving '4 * x^3 - 2 * x - 5' using incremental_search method:
dx = 0.1; Interval: [1.2000000000000002, 1.3000000000000003]
dx = 0.01; Interval: [1.2300000000000002, 1.2400000000000002]
dx = 0.001; Interval: [1.231, 1.232]
dx = 0.0001; Interval: [1.231, 1.2311]

Solution: 1.2310500000000002


### 2. Bisection Method

Bisection method is a root-finding method that repeatedly bisects an interval and selects a subinterval in which a root must lie.

Formula:

- Suppose we have a continuous function $f(x)$ defined on an interval $[a, b]$ where $f(a)$ and $f(b)$ have opposite signs. First, we check that $f(a) \cdot f(b) < 0$ to ensure that a root exists in the interval. This also determines the sign pattern of the endpoints.

- We later compute the midpoint $c = \frac{a + b}{2}$ and evaluate $f(c)$. Then determine the subinterval $[a, c]$ or $[c, b]$ where the sign change occurs and repeat the process until the desired tolerance is achieved.

- The general rule for the bisection method is to check the sign of $f(c)$ and update the interval accordingly:

    - If $f(a) \cdot f(c) < 0$, then the root lies in $[a, c]$, so we set $b = c$.
    - If $f(c) \cdot f(b) < 0$, then the root lies in $[c, b]$, so we set $a = c$.

In [526]:
def bisection(f, a, b, tol=1.0e-9):  # With recursion

    if np.sign(f(a)) == np.sign(f(b)):
        raise Exception(f"No solution found")

    # Compute the mean value
    c = (a + b) / 2.0

    if np.abs(f(c)) < tol:
        # c is the solution
        return c

    elif np.sign(f(a)) == np.sign(f(c)):
        # Solutions in range [c, b]
        print(f"c = {c} {f(c)} -> [{c}, {b}]")
        return bisection(f, c, b, tol)

    elif np.sign(f(b)) == np.sign(f(c)):
        # Solutions in range [a, c]
        print(f"c = {c}: {f(c)} -> [{a}, {c}]")
        return bisection(f, a, c, tol)

In [527]:
# Problem 1
fx = "x^2 - 5"
a, b = 2, 3

# Find the solutions
solve(fx, a, b, dx, bisection)

Solving 'x^2 - 5' using bisection method:
c = 2.5: 1.25 -> [2, 2.5]
c = 2.25: 0.0625 -> [2, 2.25]
c = 2.125 -0.484375 -> [2.125, 2.25]
c = 2.1875 -0.21484375 -> [2.1875, 2.25]
c = 2.21875 -0.0771484375 -> [2.21875, 2.25]
c = 2.234375 -0.007568359375 -> [2.234375, 2.25]
c = 2.2421875: 0.02740478515625 -> [2.234375, 2.2421875]
c = 2.23828125: 0.0099029541015625 -> [2.234375, 2.23828125]
c = 2.236328125: 0.001163482666015625 -> [2.234375, 2.236328125]
c = 2.2353515625 -0.0032033920288085938 -> [2.2353515625, 2.236328125]
c = 2.23583984375 -0.001020193099975586 -> [2.23583984375, 2.236328125]

Solution: 2.236083984375


In [528]:
# Problem 2
fx = "np.cos(x) - x"
a, b = 0, 1

# Find the solutions
solve(fx, a, b, dx, bisection)

Solving 'np.cos(x) - x' using bisection method:
c = 0.5 0.37758256189037276 -> [0.5, 1]
c = 0.75: -0.018311131126179103 -> [0.5, 0.75]
c = 0.625 0.18596311950521793 -> [0.625, 0.75]
c = 0.6875 0.0853349461524715 -> [0.6875, 0.75]
c = 0.71875 0.03387937241806649 -> [0.71875, 0.75]
c = 0.734375 0.00787472545850132 -> [0.734375, 0.75]
c = 0.7421875: -0.005195711743759213 -> [0.734375, 0.7421875]
c = 0.73828125 0.001345149751805108 -> [0.73828125, 0.7421875]
c = 0.740234375: -0.001923872780897673 -> [0.73828125, 0.740234375]
c = 0.7392578125: -0.0002890091467900868 -> [0.73828125, 0.7392578125]
c = 0.73876953125 0.0005281584336581657 -> [0.73876953125, 0.7392578125]
c = 0.739013671875 0.00011959667132188656 -> [0.739013671875, 0.7392578125]

Solution: 0.7391357421875


In [529]:
# Problem 3
fx = "x^3 - 2 * x - 5"
a, b = 2, 3

# Find the solutions
solve(fx, a, b, dx, bisection)

Solving 'x^3 - 2 * x - 5' using bisection method:
c = 2.5: 5.625 -> [2, 2.5]
c = 2.25: 1.890625 -> [2, 2.25]
c = 2.125: 0.345703125 -> [2, 2.125]
c = 2.0625 -0.351318359375 -> [2.0625, 2.125]
c = 2.09375 -0.008941650390625 -> [2.09375, 2.125]
c = 2.109375: 0.16683578491210938 -> [2.09375, 2.109375]
c = 2.1015625: 0.07856225967407227 -> [2.09375, 2.1015625]
c = 2.09765625: 0.03471428155899048 -> [2.09375, 2.09765625]
c = 2.095703125: 0.012862332165241241 -> [2.09375, 2.095703125]
c = 2.0947265625: 0.00195434782654047 -> [2.09375, 2.0947265625]
c = 2.09423828125 -0.003495149197988212 -> [2.09423828125, 2.0947265625]
c = 2.094482421875 -0.0007707752083661035 -> [2.094482421875, 2.0947265625]
c = 2.0946044921875: 0.000591692672969657 -> [2.094482421875, 2.0946044921875]

Solution: 2.09454345703125


In [530]:
# Problem 4
fx = "x^3 - 2 * x - 3"
a, b = 1, 4

# Find the solutions
solve(fx, a, b, dx, bisection)

Solving 'x^3 - 2 * x - 3' using bisection method:
c = 2.5: 7.625 -> [1, 2.5]
c = 1.75 -1.140625 -> [1.75, 2.5]
c = 2.125: 2.345703125 -> [1.75, 2.125]
c = 1.9375: 0.398193359375 -> [1.75, 1.9375]
c = 1.84375 -0.419830322265625 -> [1.84375, 1.9375]
c = 1.890625 -0.023281097412109375 -> [1.890625, 1.9375]
c = 1.9140625: 0.18430185317993164 -> [1.890625, 1.9140625]
c = 1.90234375: 0.07972663640975952 -> [1.890625, 1.90234375]
c = 1.896484375: 0.02802743762731552 -> [1.890625, 1.896484375]
c = 1.8935546875: 0.002324412576854229 -> [1.890625, 1.8935546875]
c = 1.89208984375 -0.010490522370673716 -> [1.89208984375, 1.8935546875]
c = 1.892822265625 -0.004086101063876413 -> [1.892822265625, 1.8935546875]
c = 1.8931884765625 -0.0008816059325909009 -> [1.8931884765625, 1.8935546875]
c = 1.89337158203125: 0.0007212128814444441 -> [1.8931884765625, 1.89337158203125]

Solution: 1.893280029296875


In [531]:
# Problem 5
fx = "np.exp(x) - 3"
a, b = 0, 2

# Find the solutions
solve(fx, a, b, dx, bisection)

Solving 'np.exp(x) - 3' using bisection method:
c = 1.0 -0.2817181715409549 -> [1.0, 2]
c = 1.5: 1.4816890703380645 -> [1.0, 1.5]
c = 1.25: 0.4903429574618414 -> [1.0, 1.25]
c = 1.125: 0.08021684891803105 -> [1.0, 1.125]
c = 1.0625 -0.10640405582823886 -> [1.0625, 1.125]
c = 1.09375 -0.014551460634644187 -> [1.09375, 1.125]
c = 1.109375: 0.03246251296382807 -> [1.09375, 1.109375]
c = 1.1015625: 0.008863702464446455 -> [1.09375, 1.1015625]
c = 1.09765625 -0.002866745426236772 -> [1.09765625, 1.1015625]
c = 1.099609375: 0.0029927507631666295 -> [1.09765625, 1.099609375]

Solution: 1.0986328125


### 3. False Position Method

False position method, also known as the regula falsi method, is similar to the bisection method but uses a linear interpolation to find a better approximation of the root.

Formula:

- Suppose we have a continuous function $f(x)$ defined on an interval $[a, b]$ where $f(a)$ and $f(b)$ have opposite signs. Like the bisection method, we check that $f(a) \cdot f(b) < 0$ to ensure that a root exists in the interval.

- Then we compute the point $c$ using the formula:

$$
c = b - \frac{f(b) \cdot (a - b)}{f(a) - f(b)}
$$

- This value of $c$ is the x-intercept of the line connecting the points $(a, f(a))$ and $(b, f(b))$. We then evaluate $f(c)$ and determine the subinterval $[a, c]$ or $[c, b]$ where the sign change occurs and repeat the process until the desired tolerance is achieved.

In [532]:
def false_position(func, a, b, tol_accept):
    fa = func(a)
    fb = func(b)

    if fa * fb >= 0:
        print("The method is not valid at this stage "
              "since no root or multiple roots present, or interval does not contain a root.")
        return None

    c_before = a
    c_after = 0.0 # Initialize with a float
    tol = float('inf')  # Initial tolerance to infinity

    # Add a counter to prevent infinite loops in case of convergence issues
    max_iterations = 1000
    iterations = 0

    while tol > tol_accept and iterations < max_iterations:
        # Calculate c using the secant formula
        c_after = (a * fb - b * fa) / (fb - fa)
        fc = func(c_after)
        print(f"c: {c_after} -> f(c): {fc}")

        # Update the interval based on the sign of f(c)
        if fc * fa < 0:
            b, fb = c_after, fc
            print(f"+ Interval: [{a}, {c_after}]")
        else:
            a, fa = c_after, fc
            print(f"+ Interval: [{c_after}, {b}]")

        # Compute the tolerance
        tol = abs(c_after - c_before)
        c_before = c_after  # Update c_before
        iterations += 1

    if iterations == max_iterations:
        print("Warning: Maximum iterations reached without convergence to tolerance.")

    return c_after  # Return the root

In [533]:
# Problem 1
fx = "x^2 - 4"
a, b = 1, 3

# Find the solutions
solve(fx, a, b, dx, false_position)

Solving 'x^2 - 4' using false_position method:
c: 1.75 -> f(c): -0.9375
+ Interval: [1.75, 3]
c: 1.9473684210526316 -> f(c): -0.20775623268698018
+ Interval: [1.9473684210526316, 3]
c: 1.989361702127659 -> f(c): -0.042440018107742894
+ Interval: [1.989361702127659, 3]
c: 1.9978678038379527 -> f(c): -0.008524238387715766
+ Interval: [1.9978678038379527, 3]
c: 1.9995733788395902 -> f(c): -0.0017063026360246702
+ Interval: [1.9995733788395902, 3]
c: 1.9999146684870721 -> f(c): -0.0003413187702445697
+ Interval: [1.9999146684870721, 3]
c: 1.9999829334061507 -> f(c): -6.826608412824342e-05
+ Interval: [1.9999829334061507, 3]

Solution: 1.9999829334061507


In [534]:
# Problem 2
fx = "np.exp(x) - 3 * x"
a, b = 0, 1

# Find the solutions
solve(fx, a, b, dx, false_position)

Solving 'np.exp(x) - 3 * x' using false_position method:
c: 0.7802027171056979 -> f(c): -0.15869361924908532
+ Interval: [0, 0.7802027171056979]
c: 0.6733468659396985 -> f(c): -0.0592517494296696
+ Interval: [0, 0.6733468659396985]
c: 0.6356816179933118 -> f(c): -0.018736045818029234
+ Interval: [0, 0.6356816179933118]
c: 0.6239905033328523 -> f(c): -0.005610589004689359
+ Interval: [0, 0.6239905033328523]
c: 0.6205090819006308 -> f(c): -0.0016526163100212266
+ Interval: [0, 0.6205090819006308]
c: 0.61948531037289 -> f(c): -0.00048441407519739244
+ Interval: [0, 0.61948531037289]
c: 0.619185368265346 -> f(c): -0.00014178807704690044
+ Interval: [0, 0.619185368265346]
c: 0.6190975876088946 -> f(c): -4.1483999825286944e-05
+ Interval: [0, 0.6190975876088946]

Solution: 0.6190975876088946


In [535]:
# Problem 3
fx = "np.sin(x) - 0.5 * x"
a, b = 1, 2

# Find the solutions
solve(fx, a, b, dx, false_position)

Solving 'np.sin(x) - 0.5 * x' using false_position method:
c: 1.7901246582558137 -> f(c): 0.08098147803128775
+ Interval: [1.7901246582558137, 2]
c: 1.8891205478414554 -> f(c): 0.0052009550989279285
+ Interval: [1.8891205478414554, 2]
c: 1.8951336639640162 -> f(c): 0.00029528041413617867
+ Interval: [1.8951336639640162, 2]
c: 1.8954739464422568 -> f(c): 1.6642826650592468e-05
+ Interval: [1.8954739464422568, 2]
c: 1.8954931221920759 -> f(c): 9.376506857616818e-07
+ Interval: [1.8954931221920759, 2]

Solution: 1.8954931221920759


In [536]:
# Problem 4
fx = "np.log(x) + x - 5"
a, b = 1, 4

# Find the solutions
solve(fx, a, b, dx, false_position)

Solving 'np.log(x) + x - 5' using false_position method:
c: 3.735794502614323 -> f(c): 0.05375501681455397
+ Interval: [1, 3.735794502614323]
c: 3.6995163657068884 -> f(c): 0.007718464842401573
+ Interval: [1, 3.6995163657068884]
c: 3.6943173672385625 -> f(c): 0.001113159574939182
+ Interval: [1, 3.6943173672385625]
c: 3.693567774548816 -> f(c): 0.00016064204867127785
+ Interval: [1, 3.693567774548816]
c: 3.6934596038316228 -> f(c): 2.3184661302799725e-05
+ Interval: [1, 3.6934596038316228]
c: 3.6934439921849482 -> f(c): 3.3461700592596344e-06
+ Interval: [1, 3.6934439921849482]

Solution: 3.6934439921849482


In [537]:
# Problem 5
fx = "x^3 + 3 * x^2 - 1"
a, b = 0, 1

# Find the solutions
solve(fx, a, b, dx, false_position)

Solving 'x^3 + 3 * x^2 - 1' using false_position method:
c: 0.25 -> f(c): -0.796875
+ Interval: [0.25, 1]
c: 0.4074074074074074 -> f(c): -0.43443580754966216
+ Interval: [0.4074074074074074, 1]
c: 0.4823668639053254 -> f(c): -0.18973056928335064
+ Interval: [0.4823668639053254, 1]
c: 0.5131565583506572 -> f(c): -0.07488170122585003
+ Interval: [0.5131565583506572, 1]
c: 0.5250125153218854 -> f(c): -0.028372102412677513
+ Interval: [0.5250125153218854, 1]
c: 0.5294625607932761 -> f(c): -0.010583631072387245
+ Interval: [0.5294625607932761, 1]
c: 0.5311167233320314 -> f(c): -0.003925031603712492
+ Interval: [0.5311167233320314, 1]
c: 0.5317293823231886 -> f(c): -0.0014524809531577532
+ Interval: [0.5317293823231886, 1]
c: 0.5319559906594574 -> f(c): -0.0005370680122560589
+ Interval: [0.5319559906594574, 1]
c: 0.5320397661503269 -> f(c): -0.00019852685543386706
+ Interval: [0.5320397661503269, 1]

Solution: 0.5320397661503269


### 4. Secant Method

Secant method is an iterative root-finding algorithm that uses a sequence of roots of secant lines to approximate a root of a function.

Formula:

- Suppose we have two initial approximations $x_0$ and $x_1$ such that $f(x_0)$ and $f(x_1)$ are not equal. We compute the next approximation $x_2$ using the formula:

$$
x_2 = x_1 - f(x_1) \cdot \frac{x_1 - x_0}{f(x_1) - f(x_0)}
$$

- We then update the values of $x_0$ and $x_1$ to $x_1$ and $x_2$, respectively, and repeat the process until the desired tolerance is achieved.

In [538]:
def secant(func, a, b, tol, maximum=1000):
    x1 = a
    x2 = b
    # Add a counter to prevent infinite loops in case of convergence issues
    iterations = 0

    print(f"Initial guesses: x1={x1}, x2={x2}")

    while iterations < maximum:
        fx1 = func(x1)
        fx2 = func(x2)

        # If either function value is below the tolerance
        if abs(fx2) < tol:
            return x2

        # Division by zero check
        if fx2 == fx1:
            print("Division by zero: f(x2) == f(x1). Cannot continue secant method.")
            return None

        # Use the Secant method formula
        x3 = x2 - fx2 * ((x2 - x1) / (fx2 - fx1))

        print(f"Iteration {iterations+1}: x1={x1}, x2={x2}, x3={x3}, f(x3)={func(x3)}")

        # Check for convergence based on the change in x
        if abs(x3 - x2) < tol:
            return x3

        # Update values for the next iteration
        x1 = x2
        x2 = x3
        iterations += 1

    print(f"Warning: Maximum iterations ({maximum}) reached without convergence to tolerance {tol}.")
    return x2  # Return the last approximation

In [539]:
# Problem 1
fx = "x^2 - 4"
a, b = 1, 3

# Find the solutions
solve(fx, a, b, dx, secant)

Solving 'x^2 - 4' using secant method:
Initial guesses: x1=1, x2=3
Iteration 1: x1=1, x2=3, x3=1.75, f(x3)=-0.9375
Iteration 2: x1=3, x2=1.75, x3=1.9473684210526316, f(x3)=-0.20775623268698018
Iteration 3: x1=1.75, x2=1.9473684210526316, x3=2.00355871886121, f(x3)=0.014247539924773456
Iteration 4: x1=1.9473684210526316, x2=2.00355871886121, x3=1.9999525931544515, f(x3)=-0.00018962513478504306
Iteration 5: x1=2.00355871886121, x2=1.9999525931544515, x3=1.9999999578600827, f(x3)=-1.6855966755713325e-07

Solution: 1.9999999578600827


In [540]:
# Problem 2
fx = "x^3 - 2 * x - 5"
a, b = 2, 3

# Find the solutions
solve(fx, a, b, dx, secant)

Solving 'x^3 - 2 * x - 5' using secant method:
Initial guesses: x1=2, x2=3
Iteration 1: x1=2, x2=3, x3=2.0588235294117645, f(x3)=-0.39079991858335283
Iteration 2: x1=3, x2=2.0588235294117645, x3=2.081263659845023, f(x3)=-0.14720405955375426
Iteration 3: x1=2.0588235294117645, x2=2.081263659845023, x3=2.0948241460940524, f(x3)=0.003043795598889787
Iteration 4: x1=2.081263659845023, x2=2.0948241460940524, x3=2.0945494310352473, f(x3)=-2.2886580653747046e-05

Solution: 2.0945494310352473


In [541]:
# Problem 3
fx = "np.sin(x) - (x / 2)"
a, b = 1, 2

# Find the solutions
solve(fx, a, b, dx, secant)

Solving 'np.sin(x) - (x / 2)' using secant method:
Initial guesses: x1=1, x2=2
Iteration 1: x1=1, x2=2, x3=1.7901246582558137, f(x3)=0.08098147803128775
Iteration 2: x1=2, x2=1.7901246582558137, x3=1.8891205478414554, f(x3)=0.0052009550989279285
Iteration 3: x1=1.7901246582558137, x2=1.8891205478414554, x3=1.8959148157719223, f(x3)=-0.000344522694784799
Iteration 4: x1=1.8891205478414554, x2=1.8959148157719223, x3=1.8954927097634646, f(x3)=1.2754384798308038e-06

Solution: 1.8954927097634646


In [542]:
# Problem 4
fx = "x^2 - 5 * x + 6"
a, b = 1, 2

# Find the solutions
solve(fx, a, b, dx, secant)

Solving 'x^2 - 5 * x + 6' using secant method:
Initial guesses: x1=1, x2=2

Solution: 2


In [543]:
# Problem 5
fx = "np.exp(x) - 5 * (np.exp(x) - 5)"
a, b = 1, 2

# Find the solutions
solve(fx, a, b, dx, secant)

Solving 'np.exp(x) - 5 * (np.exp(x) - 5)' using secant method:
Initial guesses: x1=1, x2=2
Iteration 1: x1=1, x2=2, x3=1.7561312037424492, f(x3)=1.8400251944175459
Iteration 2: x1=2, x2=1.7561312037424492, x3=1.8262855532440516, f(x3)=0.15690331969328852
Iteration 3: x1=1.7561312037424492, x2=1.8262855532440516, x3=1.8328254541823172, f(x3)=-0.006100505052351934
Iteration 4: x1=1.8262855532440516, x2=1.8328254541823172, x3=1.8325806949027235, f(x3)=1.9221132276925346e-05

Solution: 1.8325806949027235


In [544]:
# Problem 6
fx = "x^3 - 3 * x + 1"
a, b = 1, 3

# Find the solutions
solve(fx, a, b, dx, secant)

Solving 'x^3 - 3 * x + 1' using secant method:
Initial guesses: x1=1, x2=3
Iteration 1: x1=1, x2=3, x3=1.0999999999999999, f(x3)=-0.9690000000000003
Iteration 2: x1=3, x2=1.0999999999999999, x3=1.1921979067554709, f(x3)=-0.8820800964445532
Iteration 3: x1=1.0999999999999999, x2=1.1921979067554709, x3=2.1278402068067264, f(x3)=4.250709879764744
Iteration 4: x1=1.1921979067554709, x2=2.1278402068067264, x3=1.3529898842359773, f(x3)=-0.582211229318566
Iteration 5: x1=2.1278402068067264, x2=1.3529898842359773, x3=1.446334374795512, f(x3)=-0.3134406545027386
Iteration 6: x1=1.3529898842359773, x2=1.446334374795512, x3=1.555192862132729, f(x3)=0.09584949850287972
Iteration 7: x1=1.446334374795512, x2=1.555192862132729, x3=1.529699868122411, f(x3)=-0.009629927067742017
Iteration 8: x1=1.555192862132729, x2=1.529699868122411, x3=1.5320272952175973, f(x3)=-0.00024892663626063793
Iteration 9: x1=1.529699868122411, x2=1.5320272952175973, x3=1.5320890539396204, f(x3)=6.778316521405259e-07

Solutio

### 5. Ridder's Method

Ridder's method is a root-finding algorithm that combines the bisection method and the secant method to achieve faster convergence.

Formula:

- Suppose we have a continuous function $f(x)$ defined on an interval $[a, b]$ where $f(a)$ and $f(b)$ have opposite signs. We first compute the midpoint $c = \frac{a + b}{2}$ and evaluate $f(c)$.

- We then compute the value of $d$ using the formula:

$$
d = \sqrt{f(c)^2 - f(a) \cdot f(b)}
$$

- We then compute the next approximation $x$ using the formula:

$$
x = c + \frac{(c - a) \cdot f(c)}{d}
$$

- We then evaluate $f(x)$ and determine the subinterval $[a, x]$ or $[x, b]$ where the sign change occurs and repeat the process until the desired tolerance is achieved.

In [545]:
def ridder(func, x1, x2, tol, max_iterations=1000):
    x4_before = 0

    for i in range(max_iterations):
        print(f"Iteration {i + 1}")
        # Find root x based on the Ridder's formula
        x3 = 0.5 * (x1 + x2)
        f1 = func(x1)
        f2 = func(x2)
        f3 = func(x3)

        # Check for invalid discriminant
        discriminant = f3 ** 2 - f1 * f2
        if discriminant <= 0:
            print("Warning: Discriminant is non-positive. Method may not converge or interval is invalid.")
            return x3 # Return current midpoint as best guess

        s = math.sqrt(discriminant)

        # The choice of using + or - depends on sign (standard Ridder's method choice)
        # If f1 and f3 have different signs, the root is in (x1, x3)
        # If f2 and f3 have different signs, the root is in (x3, x2)
        # The formula for x4 involves f1, f2, f3 and their signs.
        # This is a common way to implement the Ridder's step:
        if np.sign(f1) != np.sign(f3):
            x4 = x3 - (x3 - x1) * f3 / s
        elif np.sign(f2) != np.sign(f3):
            x4 = x3 + (x2 - x3) * f3 / s
        else:
            # This case should ideally not happen if a root is bracketed
            print("Warning: Cannot determine proper interval for next step.")
            return x3

        # Update f4
        x4_after = x4
        print(f"Difference between two successive values of x4: {abs(x4_before - x4_after)}")
        if abs(x4_before - x4_after) < tol:
            print(f"The algorithm converged")
            break
        # Assign the f4 for the next iteration
        x4_before = x4_after

        # Update the interval (x1, x2) for the next iteration based on x4
        f4 = func(x4)
        if np.sign(f3) != np.sign(f4):
            x1, x2 = x3, x4
        elif np.sign(f1) != np.sign(f4):
            x2 = x4
        else:
            x1 = x4

        print(f"New Interval: {x1, x2}")

    # Return the midpoint as the approximate result
    return x4

In [546]:
# Problem 1
fx = "x^3 - x - 1"
a, b = 0, 2

# Find the solutions
solve(fx, a, b, dx, ridder)

Solving 'x^3 - x - 1' using ridder method:
Iteration 1
Difference between two successive values of x4: 0.5917517095361369
New Interval: (0.5917517095361369, 2)
Iteration 2
Difference between two successive values of x4: 0.6721185583076172
New Interval: (1.2638702678437541, 2)
Iteration 3
Difference between two successive values of x4: 0.05884991935420336
New Interval: (1.631935133921877, 1.3227201871979575)
Iteration 4
Difference between two successive values of x4: 0.001981149463323728
New Interval: (1.4773276605599173, 1.3247013366612812)
Iteration 5
Difference between two successive values of x4: 1.658414765137728e-05
The algorithm converged

Solution: 1.3247179208089326


In [547]:
# Problem 2
fx = "2 * x^3 - 2 * x - 5"
a, b = 1, 2

# Find the solutions
solve(fx, a, b, dx, ridder)

Solving '2 * x^3 - 2 * x - 5' using ridder method:
Iteration 1
Difference between two successive values of x4: 1.3966377211756595
New Interval: (1.3966377211756595, 2)
Iteration 2
Difference between two successive values of x4: 0.20313215660850514
New Interval: (1.6983188605878299, 1.5997698777841647)
Iteration 3
Difference between two successive values of x4: 0.0008282513345565512
New Interval: (1.6490443691859973, 1.6005981291187212)
Iteration 4
Difference between two successive values of x4: 4.1576260478848326e-07
The algorithm converged

Solution: 1.600598544881326


In [548]:
# Problem 3
fx = "x^3 + 2 * x^2 + x - 1"
a, b = 0, 4

# Find the solutions
solve(fx, a, b, dx, ridder)

Solving 'x^3 + 2 * x^2 + x - 1' using ridder method:
Iteration 1
Difference between two successive values of x4: 0.2739115192728474
New Interval: (2.0, 0.2739115192728474)
Iteration 2
Difference between two successive values of x4: 0.16698974093834162
New Interval: (1.1369557596364237, 0.44090126021118903)
Iteration 3
Difference between two successive values of x4: 0.023848240231098217
New Interval: (0.7889285099238064, 0.46474950044228724)
Iteration 4
Difference between two successive values of x4: 0.0008145021442503841
New Interval: (0.6268390051830468, 0.46556400258653763)
Iteration 5
Difference between two successive values of x4: 7.21273650861054e-06
The algorithm converged

Solution: 0.46557121532304624


### 6. Newton-Raphson Method

Newton-Raphson method is an iterative root-finding algorithm that uses the derivative of a function to find its roots.

Formula:

- Suppose we have a function $f(x)$ and its derivative $f'(x)$. We start with an initial guess $x_0$ and compute the next approximation $x_1$ using the formula:

$$
x_1 = x_0 - \frac{f(x_0)}{f'(x_0)}
$$

- We then update the value of $x_0$ to $x_1$ and repeat the process until the desired tolerance is achieved.

- The general formula for the Newton-Raphson method is:

$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}
$$

- The method converges quadratically, meaning that the number of correct digits approximately doubles with each iteration, making it very efficient for finding roots when the initial guess is close to the actual root.

In [549]:
def newton_raphson(func, df_func, x0, tol=1.0e-9, max_iterations=100):
    x_current = x0
    print(f"Initial guess for Newton-Raphson: x0={x_current}")

    for i in range(max_iterations):
        f_val = func(x_current)
        df_val = df_func(x_current)

        print(f"  Iteration {i+1}: x={x_current:.6f}, f(x)={f_val:.6f}, f'(x)={df_val:.6f}")

        if abs(f_val) < tol:
            print(f"  Converged: |f(x)| < tolerance.")
            return x_current

        if df_val == 0:
            print(f"  Division by zero: derivative is zero at x={x_current}. Cannot continue Newton-Raphson.")
            return None

        x_next = x_current - f_val / df_val

        if abs(x_next - x_current) < tol:
            print(f"  Converged: |x_next - x_current| < tolerance.")
            return x_next

        x_current = x_next

    print(f"  Warning: Maximum iterations ({max_iterations}) reached without convergence to tolerance {tol}.")
    return x_current # Return the last approximation

def derivative(f):
    x = sym.symbols('x')
    # Convert the string to a symbolic expression
    fx = sym.sympify(f)
    derivative_fx = sym.diff(fx, x)
    return derivative_fx

In [550]:
# Problem 1
fx = "x^2 - 2"
x0 = 1

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'x^2 - 2' using newton_raphson method:
Initial guess for Newton-Raphson: x0=1
  Iteration 1: x=1.000000, f(x)=-1.000000, f'(x)=2.000000
  Iteration 2: x=1.500000, f(x)=0.250000, f'(x)=3.000000
  Iteration 3: x=1.416667, f(x)=0.006944, f'(x)=2.833333
  Iteration 4: x=1.414216, f(x)=0.000006, f'(x)=2.828431
  Converged: |f(x)| < tolerance.

Solution: 1.4142156862745099


In [551]:
# Problem 2
fx = "x^3 - 2 * x + 1"
x0 = 0

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'x^3 - 2 * x + 1' using newton_raphson method:
Initial guess for Newton-Raphson: x0=0
  Iteration 1: x=0.000000, f(x)=1.000000, f'(x)=-2.000000
  Iteration 2: x=0.500000, f(x)=0.125000, f'(x)=-1.250000
  Iteration 3: x=0.600000, f(x)=0.016000, f'(x)=-0.920000
  Iteration 4: x=0.617391, f(x)=0.000550, f'(x)=-0.856484
  Iteration 5: x=0.618033, f(x)=0.000001, f'(x)=-0.854105
  Converged: |f(x)| < tolerance.

Solution: 0.6180330952207308


In [552]:
# Problem 3
fx = "np.cos(x) - x"
x0 = 0.5

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'np.cos(x) - x' using newton_raphson method:
Initial guess for Newton-Raphson: x0=0.5
  Iteration 1: x=0.500000, f(x)=0.377583, f'(x)=-1.479426
  Iteration 2: x=0.755222, f(x)=-0.027103, f'(x)=-1.685451
  Iteration 3: x=0.739142, f(x)=-0.000095, f'(x)=-1.673654
  Converged: |f(x)| < tolerance.

Solution: 0.7391416661498792


In [553]:
# Problem 4
fx = "np.exp(x) - 3 * x"
x0 = 1

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'np.exp(x) - 3 * x' using newton_raphson method:
Initial guess for Newton-Raphson: x0=1
  Iteration 1: x=1.000000, f(x)=-0.281718, f'(x)=-0.281718
  Iteration 2: x=0.000000, f(x)=1.000000, f'(x)=-2.000000
  Iteration 3: x=0.500000, f(x)=0.148721, f'(x)=-1.351279
  Iteration 4: x=0.610060, f(x)=0.010362, f'(x)=-1.159459
  Iteration 5: x=0.618997, f(x)=0.000074, f'(x)=-1.142936
  Converged: |f(x)| < tolerance.

Solution: 0.6189967797415397


In [554]:
# Problem 5
fx = "x^3 - 4 * x^2 + 6"
x0 = 2

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'x^3 - 4 * x^2 + 6' using newton_raphson method:
Initial guess for Newton-Raphson: x0=2
  Iteration 1: x=2.000000, f(x)=-2.000000, f'(x)=-4.000000
  Iteration 2: x=1.500000, f(x)=0.375000, f'(x)=-5.250000
  Iteration 3: x=1.571429, f(x)=0.002915, f'(x)=-5.163265
  Iteration 4: x=1.571993, f(x)=0.000000, f'(x)=-5.162458
  Converged: |f(x)| < tolerance.

Solution: 1.5719932241671373


In [555]:
# Problem 6
fx = "np.log(x) - 1"
x0 = 2

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'np.log(x) - 1' using newton_raphson method:
Initial guess for Newton-Raphson: x0=2
  Iteration 1: x=2.000000, f(x)=-0.306853, f'(x)=0.500000
  Iteration 2: x=2.613706, f(x)=-0.039231, f'(x)=0.382599
  Iteration 3: x=2.716244, f(x)=-0.000750, f'(x)=0.368155
  Iteration 4: x=2.718281, f(x)=-0.000000, f'(x)=0.367880
  Converged: |f(x)| < tolerance.

Solution: 2.718281064358138


In [556]:
# Problem 7
fx = "x^4 - 8 * x^2 + 16"
x0 = 2.5

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'x^4 - 8 * x^2 + 16' using newton_raphson method:
Initial guess for Newton-Raphson: x0=2.5
  Iteration 1: x=2.500000, f(x)=5.062500, f'(x)=22.500000
  Iteration 2: x=2.275000, f(x)=1.382094, f'(x)=10.698187
  Iteration 3: x=2.145810, f(x)=0.365423, f'(x)=5.188591
  Iteration 4: x=2.075382, f(x)=0.094379, f'(x)=2.550324
  Iteration 5: x=2.038376, f(x)=0.024017, f'(x)=1.263590
  Iteration 6: x=2.019368, f(x)=0.006060, f'(x)=0.628822
  Iteration 7: x=2.009731, f(x)=0.001522, f'(x)=0.313657
  Iteration 8: x=2.004877, f(x)=0.000382, f'(x)=0.156639
  Iteration 9: x=2.002442, f(x)=0.000095, f'(x)=0.078272
  Converged: |f(x)| < tolerance.

Solution: 2.0024415195627774


In [557]:
# Problem 8
fx = "x * np.sin(x) - 1"
x0 = 1

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'x * np.sin(x) - 1' using newton_raphson method:
Initial guess for Newton-Raphson: x0=1
  Iteration 1: x=1.000000, f(x)=-0.158529, f'(x)=1.381773
  Iteration 2: x=1.114729, f(x)=0.000794, f'(x)=1.388741
  Iteration 3: x=1.114157, f(x)=-0.000000, f'(x)=1.388809
  Converged: |f(x)| < tolerance.

Solution: 1.1141571268362673


In [558]:
# Problem 9
fx = "x^5 - 3 * x^3 + 2"
x0 = 1

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'x^5 - 3 * x^3 + 2' using newton_raphson method:
Initial guess for Newton-Raphson: x0=1
  Iteration 1: x=1.000000, f(x)=0.000000, f'(x)=-4.000000
  Converged: |f(x)| < tolerance.

Solution: 1


In [559]:
# Problem 10
fx = "x^3 - 6 * x^2 + 11 * x - 6"
x0 = 3

# Find the solutions
solve(fx, x0, None, dx, newton_raphson)

Solving 'x^3 - 6 * x^2 + 11 * x - 6' using newton_raphson method:
Initial guess for Newton-Raphson: x0=3
  Iteration 1: x=3.000000, f(x)=0.000000, f'(x)=2.000000
  Converged: |f(x)| < tolerance.

Solution: 3


# Recap

Numerical methods for finding roots of equations are essential tools in mathematics and applied sciences. The methods discussed, including Incremental Search, Bisection, False Position, Secant, Ridder's, and Newton-Raphson, each have their own advantages and limitations. Understanding the underlying principles and appropriate applications of these methods allows for effective root-finding in various contexts.